<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-08-tool-use-and-mcp/lesson-8.5-building-mcp-servers/notebooks/exercises-8_5.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 8.5: Building MCP Servers & Clients

*Module 8 · MCP, Tool Orchestration & Function Calling CLASS*

> Lesson 8.4 explained the protocol. Now we BUILD. This lesson creates a production-quality MCP server exposing all three primitives — tools, resources, and prompts — plus a client that discovers and consumes them. You’ll build a complete Netsetos support server: course search tools, knowledge base resources, and support prompt templates. Then connect it to Claude Desktop and Gemini.

`MCP Server` · `All 3 Primitives` · `Client SDK` · `Claude Desktop` · `75 min`

---

## About this notebook

This notebook contains the **3 runnable code examples** from the published lesson `lesson-8.5.html`. Each block is reproduced verbatim — every `#` comment from the source is preserved — and is preceded by a short markdown header that names the step, the file, and what the block demonstrates.

**How to use it:**

1. Run the **Setup** cells once (install + API keys).
2. Step through the lesson cells in order — each is independent enough to inspect on its own.
3. Read the *What just happened?* recap that follows each block (where the lesson provides one).


## Contents

1. Step 1 — Complete MCP Server — Tools + Resources + Prompts — `01_netsetos_server.py`
2. Step 2 — MCP Client — Discover and Consume Everything — `02_mcp_client.py`
3. Step 3 — Claude Desktop Integration — Use Your Server in Claude — `03_claude_desktop_config.json`


## Lesson code

Each section below corresponds to one code block from the published lesson.


### Step 1 · Complete MCP Server — Tools + Resources + Prompts

One server exposing all three primitives. Production-ready structure.

**`01_netsetos_server.py`** · _Block 1 of 3_

NETSETOS MCP SERVER — Tools + Resources + Prompts — pip install mcp


In [ ]:
# NETSETOS MCP SERVER — Tools + Resources + Prompts
# pip install mcp
from mcp.server import Server
from mcp.server.stdio import stdio_server
from mcp.types import (
    Tool, TextContent, Resource, ResourceTemplate,
    Prompt, PromptMessage, PromptArgument
)
import json, asyncio

app = Server("netsetos-support")

# ═══════════ KNOWLEDGE BASE ═══════════
COURSES = {
    "genai": {"name":"GenAI Complete", "price":14999, "hours":146, "rating":4.8},
    "python": {"name":"Python Mastery", "price":9999, "hours":80, "rating":4.7},
    "cloud": {"name":"GCP Cloud", "price":11999, "hours":110, "rating":4.6},
}

# ═══════════ TOOLS (model-controlled) ═══════════
@app.list_tools()
async def list_tools():
    return [
        Tool(name="search_courses",
             description="Search Netsetos courses by topic. Use when user asks about available courses.",
             inputSchema={"type":"object",
                 "properties":{"topic":{"type":"string","description":"Topic: genai, python, cloud"}},
                 "required":["topic"]}),
        Tool(name="get_refund_policy",
             description="Check refund eligibility based on days since purchase.",
             inputSchema={"type":"object",
                 "properties":{"days_since_purchase":{"type":"integer"}},
                 "required":["days_since_purchase"]}),
        Tool(name="calculate_emi",
             description="Calculate monthly EMI for a course.",
             inputSchema={"type":"object",
                 "properties":{"amount":{"type":"integer"}, "months":{"type":"integer"}},
                 "required":["amount","months"]}),
    ]

@app.call_tool()
async def call_tool(name, arguments):
    if name == "search_courses":
        topic = arguments["topic"].lower()
        course = COURSES.get(topic)
        if course:
            return [TextContent("text", json.dumps(course))]
        return [TextContent("text", json.dumps({"error":f"No course for '{topic}'. Try: genai, python, cloud"}))]

    elif name == "get_refund_policy":
        days = arguments["days_since_purchase"]
        if days <= 7: policy = {"eligible":True, "refund":"100%", "message":"Full refund available"}
        elif days <= 30: policy = {"eligible":True, "refund":"50%", "message":"Partial refund available"}
        else: policy = {"eligible":False, "refund":"0%", "message":"Refund window expired"}
        return [TextContent("text", json.dumps(policy))]

    elif name == "calculate_emi":
        amt, m = arguments["amount"], arguments["months"]
        emi = round(amt / m)
        return [TextContent("text", json.dumps({"emi":emi, "total":amt, "months":m}))]

# ═══════════ RESOURCES (app-controlled) ═══════════
@app.list_resources()
async def list_resources():
    return [
        Resource(uri="netsetos://courses/catalog", name="Course Catalog",
                 description="Complete list of all courses with pricing", mimeType="application/json"),
        Resource(uri="netsetos://faq/refunds", name="Refund FAQ",
                 description="Frequently asked questions about refunds", mimeType="text/plain"),
        Resource(uri="netsetos://stats/students", name="Student Stats",
                 description="Current student enrollment statistics", mimeType="application/json"),
    ]

@app.read_resource()
async def read_resource(uri):
    if str(uri) == "netsetos://courses/catalog":
        return json.dumps(COURSES, indent=2)
    elif str(uri) == "netsetos://faq/refunds":
        return "Refund policy: Full within 7 days. 50% 8-30 days. None after 30 days. Email [email protected]."
    elif str(uri) == "netsetos://stats/students":
        return json.dumps({"total":5200, "active":3100, "completion_rate":"85%"})

# ═══════════ PROMPTS (user-controlled) ═══════════
@app.list_prompts()
async def list_prompts():
    return [
        Prompt(name="support_greeting", description="Generate a personalized support greeting",
               arguments=[PromptArgument(name="student_name", required=True)]),
        Prompt(name="course_recommendation", description="Recommend a course based on interests",
               arguments=[PromptArgument(name="interests", required=True),
                          PromptArgument(name="budget", required=False)]),
    ]

@app.get_prompt()
async def get_prompt(name, arguments):
    if name == "support_greeting":
        return PromptMessage(role="user",
            content=TextContent("text", f"Greet {arguments['student_name']} warmly. You are a Netsetos support agent. Ask how you can help with their learning journey."))
    elif name == "course_recommendation":
        budget = arguments.get("budget", "any")
        return PromptMessage(role="user",
            content=TextContent("text", f"Recommend a Netsetos course for someone interested in {arguments['interests']} with budget {budget}. Use search_courses tool to find options."))

# ═══════════ RUN ═══════════
async def main():
    async with stdio_server() as (read, write):
        await app.run(read, write, app.create_initialization_options())

if __name__ == "__main__":
    asyncio.run(main())


> **What just happened?** One server, three primitives: 3 tools (search_courses, get_refund_policy, calculate_emi), 3 resources (catalog, FAQ, stats with custom URIs), 2 prompts (greeting, recommendation with arguments). The LLM calls tools. The app reads resources. The user selects prompts. This is a complete, production-structured MCP server in ~80 lines.


### Step 2 · MCP Client — Discover and Consume Everything

Connect to the server, discover all primitives, call tools, read resources, use prompts.

**`02_mcp_client.py`** · _Block 2 of 3_

MCP CLIENT — Discover and consume all 3 primitives


In [ ]:
# MCP CLIENT — Discover and consume all 3 primitives
from mcp.client import ClientSession
from mcp.client.stdio import stdio_client, StdioServerParameters
import asyncio, json

async def run_client():
    server = StdioServerParameters(command="python", args=["01_netsetos_server.py"])

    async with stdio_client(server) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            print("Connected to MCP server!\n")

            # ── 1. Discover tools ──
            tools = await session.list_tools()
            print("TOOLS:")
            for t in tools.tools:
                print(f"  {t.name}: {t.description[:60]}")

            # ── 2. Call tools ──
            print("\nTOOL CALLS:")
            r = await session.call_tool("search_courses", {"topic":"genai"})
            print(f"  search_courses: {r.content[0].text}")

            r = await session.call_tool("get_refund_policy", {"days_since_purchase":5})
            print(f"  refund (5 days): {r.content[0].text}")

            r = await session.call_tool("calculate_emi", {"amount":14999,"months":6})
            print(f"  emi: {r.content[0].text}")

            # ── 3. Discover & read resources ──
            resources = await session.list_resources()
            print("\nRESOURCES:")
            for res in resources.resources:
                print(f"  {res.uri}: {res.name}")
                content = await session.read_resource(res.uri)
                print(f"    → {str(content)[:80]}...")

            # ── 4. Discover & use prompts ──
            prompts = await session.list_prompts()
            print("\nPROMPTS:")
            for p in prompts.prompts:
                print(f"  {p.name}: {p.description}")

            prompt = await session.get_prompt("support_greeting", {"student_name":"Priya"})
            print(f"  → {prompt.messages[0].content.text[:80]}")

if __name__ == "__main__":
    asyncio.run(run_client())


### Step 3 · Claude Desktop Integration — Use Your Server in Claude

Add your MCP server to Claude Desktop. Chat with your tools naturally.

**`03_claude_desktop_config.json`** · _Block 3 of 3_


In [ ]:
{
  "mcpServers": {
    "netsetos-support": {
      "command": "python",
      "args": ["/path/to/01_netsetos_server.py"],
      "env": {}
    }
  }
}

// Location:
// macOS: ~/Library/Application Support/Claude/claude_desktop_config.json
// Windows: %APPDATA%\Claude\claude_desktop_config.json
// Linux: ~/.config/Claude/claude_desktop_config.json
//
// After adding, restart Claude Desktop.
// You'll see your tools in the 🔨 menu.
// Ask: "Search for GenAI courses" → Claude calls search_courses
// Ask: "Can I get a refund? I bought 10 days ago" → calls get_refund_policy


> **What just happened?** MCP's power is that servers work with ANY LLM client, not just Claude. The bridge pattern converts MCP tool schemas to the target provider's function calling format. LiteLLM provides the most LLM-agnostic approach (100+ models), while LangChain adapters integrate directly with agent frameworks.


---

## Wrap-up

You've walked through all 3 code examples from this lesson. Re-run any cell to experiment — change a prompt, swap a model, raise the temperature. The published lesson page (linked at the top) has the surrounding narrative, quizzes, and practice exercises if you want to go deeper.
